# Advanced spaCy Features and Integration with Modern Tokenizers

**Objective**: Explore advanced spaCy capabilities including Named Entity Recognition, dependency parsing, and custom pipelines, while demonstrating integration with modern tokenization approaches.

**Duration**: ~40 minutes

**Learning Outcomes**:
- Master advanced spaCy features (NER, dependency parsing, custom pipelines)
- Understand when to combine classical NLP with modern approaches
- Build hybrid processing systems
- Create custom NLP pipelines for specific domains

---

## 1. Setup and Advanced spaCy Configuration

Let's start with advanced spaCy setup and explore available models and capabilities.

In [ ]:
# Advanced spaCy imports
import spacy
from spacy import displacy
from spacy.tokens import Doc, Span, Token
from spacy.matcher import Matcher, PhraseMatcher
from spacy.util import filter_spans
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter, defaultdict
import networkx as nx
from IPython.display import HTML, display
import warnings
warnings.filterwarnings('ignore')

# Load multiple language models with different capabilities
models = {}
try:
    models['en_sm'] = spacy.load("en_core_web_sm")
    print("✓ English small model loaded")
except:
    print("❌ English small model not found")

try:
    models['it_sm'] = spacy.load("it_core_news_sm")
    print("✓ Italian small model loaded")
except:
    print("❌ Italian small model not found")

# Try to load larger models if available
try:
    models['en_lg'] = spacy.load("en_core_web_lg")
    print("✓ English large model loaded (with word vectors)")
except:
    print("⚠ English large model not available - install with: python -m spacy download en_core_web_lg")

print(f"\nLoaded models: {list(models.keys())}")

# Display model information
for name, nlp in models.items():
    print(f"\n{name.upper()}:")
    print(f"  Language: {nlp.lang}")
    print(f"  Pipeline: {nlp.pipe_names}")
    print(f"  Vectors: {nlp.vocab.vectors.size} vectors" if nlp.vocab.vectors.size > 0 else "  Vectors: None")

## 2. Advanced Named Entity Recognition

Let's explore spaCy's sophisticated NER capabilities and how to customize them.

In [ ]:
# Sample texts for advanced NER analysis
complex_texts = {
    'tech_news': """
    In January 2024, OpenAI announced the release of GPT-4 Turbo, which significantly improved upon 
    the capabilities of previous models. The San Francisco-based company, led by CEO Sam Altman, 
    has raised over $10 billion in funding from Microsoft Corporation. Meanwhile, Google DeepMind, 
    headquartered in London, continues to develop its Gemini models. Meta's Chief AI Scientist Yann LeCun 
    recently spoke at the Neural Information Processing Systems (NeurIPS) conference about the future 
    of artificial intelligence. The conference, held in New Orleans from December 10-16, 2023, 
    attracted over 15,000 researchers from around the world.
    """,
    
    'financial': """
    Apple Inc. (NASDAQ: AAPL) reported record quarterly revenue of $89.5 billion in Q1 2024, 
    representing a 13% increase year-over-year. The company's stock price rose 4.2% to $185.92 
    following the earnings announcement. Tesla (TSLA) delivered 484,507 vehicles in Q4 2023, 
    falling short of Wall Street expectations of 500,000 units. The Federal Reserve's decision 
    to maintain interest rates at 5.25-5.5% has impacted market sentiment across the S&P 500.
    """,
    
    'academic': """
    Dr. Sarah Chen from Stanford University and Prof. Marco Rossi from the University of Milan 
    co-authored a groundbreaking paper published in Nature Machine Intelligence on January 15, 2024. 
    Their research on transformer architectures, funded by a €2.5 million grant from the European 
    Research Council, demonstrates novel approaches to reducing computational complexity. The study, 
    conducted between 2022 and 2023, involved collaboration with researchers from MIT, 
    Oxford University, and ETH Zurich.
    """
}

def advanced_ner_analysis(text, nlp_model, display_viz=False):
    """
    Perform comprehensive NER analysis with detailed statistics
    """
    doc = nlp_model(text)
    
    # Extract entities with detailed information
    entities = []
    for ent in doc.ents:
        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'description': spacy.explain(ent.label_),
            'start': ent.start_char,
            'end': ent.end_char,
            'confidence': getattr(ent, 'confidence', None),  # If available
            'start_token': ent.start,
            'end_token': ent.end
        })
    
    # Entity statistics
    entity_counts = Counter([ent['label'] for ent in entities])
    entity_types = len(entity_counts)
    total_entities = len(entities)
    
    # Display visualization if requested
    if display_viz and entities:
        try:
            html = displacy.render(doc, style="ent", jupyter=False)
            display(HTML(html))
        except:
            print("Visualization not available")
    
    return {
        'entities': entities,
        'entity_counts': dict(entity_counts),
        'total_entities': total_entities,
        'entity_types': entity_types,
        'doc': doc
    }

# Analyze each text type
if 'en_sm' in models:
    nlp = models['en_sm']
    
    for text_type, text in complex_texts.items():
        print(f"\n{'='*60}")
        print(f"ANALYZING: {text_type.upper()}")
        print(f"{'='*60}")
        
        analysis = advanced_ner_analysis(text, nlp, display_viz=True)
        
        print(f"\nSTATISTICS:")
        print(f"  Total entities: {analysis['total_entities']}")
        print(f"  Entity types: {analysis['entity_types']}")
        
        print(f"\nENTITY BREAKDOWN:")
        for label, count in analysis['entity_counts'].items():
            description = spacy.explain(label) or "Unknown"
            print(f"  {label}: {count} ({description})")
        
        print(f"\nDETAILED ENTITIES:")
        for ent in analysis['entities'][:10]:  # Show first 10
            print(f"  '{ent['text']}' -> {ent['label']} ({ent['description']})")
        
        if len(analysis['entities']) > 10:
            print(f"  ... and {len(analysis['entities']) - 10} more entities")
else:
    print("English model not available for NER analysis")

## 3. Dependency Parsing and Syntactic Analysis

Explore spaCy's dependency parsing capabilities for understanding sentence structure.

In [ ]:
def analyze_dependencies(text, nlp_model, show_viz=False):
    """
    Comprehensive dependency parsing analysis
    """
    doc = nlp_model(text)
    
    # Extract dependency information
    dependencies = []
    for token in doc:
        dependencies.append({
            'text': token.text,
            'lemma': token.lemma_,
            'pos': token.pos_,
            'tag': token.tag_,
            'dep': token.dep_,
            'head_text': token.head.text,
            'head_pos': token.head.pos_,
            'children': [child.text for child in token.children],
            'is_alpha': token.is_alpha,
            'is_stop': token.is_stop,
            'is_punct': token.is_punct
        })
    
    # Dependency statistics
    dep_counts = Counter([dep['dep'] for dep in dependencies if not dep['is_punct']])
    pos_counts = Counter([dep['pos'] for dep in dependencies if not dep['is_punct']])
    
    # Find root and main clauses
    roots = [dep for dep in dependencies if dep['dep'] == 'ROOT']
    subjects = [dep for dep in dependencies if 'subj' in dep['dep']]
    objects = [dep for dep in dependencies if 'obj' in dep['dep']]
    
    # Visualization
    if show_viz:
        try:
            # Limit to first sentence for better visualization
            first_sent = list(doc.sents)[0] if list(doc.sents) else doc
            html = displacy.render(first_sent, style="dep", jupyter=False, options={'compact': True})
            display(HTML(html))
        except:
            print("Dependency visualization not available")
    
    return {
        'dependencies': dependencies,
        'dep_counts': dict(dep_counts),
        'pos_counts': dict(pos_counts),
        'roots': roots,
        'subjects': subjects,
        'objects': objects,
        'sentences': len(list(doc.sents))
    }

# Test dependency parsing
sample_sentences = [
    "The advanced neural network processes natural language with remarkable accuracy.",
    "OpenAI's GPT-4 model, which was trained on diverse datasets, demonstrates impressive capabilities.",
    "Researchers are developing new techniques to improve model efficiency and reduce computational costs."
]

if 'en_sm' in models:
    nlp = models['en_sm']
    
    for i, sentence in enumerate(sample_sentences, 1):
        print(f"\n{'='*80}")
        print(f"SENTENCE {i}: {sentence}")
        print(f"{'='*80}")
        
        dep_analysis = analyze_dependencies(sentence, nlp, show_viz=True)
        
        print(f"\nSYNTACTIC STRUCTURE:")
        print(f"  Sentences: {dep_analysis['sentences']}")
        print(f"  Roots: {[r['text'] for r in dep_analysis['roots']]}")
        print(f"  Subjects: {[s['text'] for s in dep_analysis['subjects']]}")
        print(f"  Objects: {[o['text'] for o in dep_analysis['objects']]}")
        
        print(f"\nDEPENDENCY PATTERNS:")
        for dep_type, count in sorted(dep_analysis['dep_counts'].items(), key=lambda x: x[1], reverse=True)[:8]:
            description = spacy.explain(dep_type) or "Unknown relation"
            print(f"  {dep_type}: {count} ({description})")
        
        print(f"\nPOS DISTRIBUTION:")
        for pos, count in sorted(dep_analysis['pos_counts'].items(), key=lambda x: x[1], reverse=True)[:6]:
            description = spacy.explain(pos) or "Unknown POS"
            print(f"  {pos}: {count} ({description})")
else:
    print("English model not available for dependency parsing")

## 4. Custom Matchers and Pattern Recognition

Learn to create custom patterns for extracting specific information.

In [ ]:
if 'en_sm' in models:
    nlp = models['en_sm']
    
    # Initialize matchers
    matcher = Matcher(nlp.vocab)
    phrase_matcher = PhraseMatcher(nlp.vocab)
    
    # Define patterns for technical terms
    ai_patterns = [
        [{"LOWER": "artificial"}, {"LOWER": "intelligence"}],
        [{"LOWER": "machine"}, {"LOWER": "learning"}],
        [{"LOWER": "deep"}, {"LOWER": "learning"}],
        [{"LOWER": "neural"}, {"LOWER": "network"}, {"LOWER": "?", "OP": "?"}],
        [{"LOWER": "natural"}, {"LOWER": "language"}, {"LOWER": "processing"}],
        [{"LOWER": "large"}, {"LOWER": "language"}, {"LOWER": "model"}, {"LOWER": "?", "OP": "?"}]
    ]
    
    # Add patterns to matcher
    matcher.add("AI_TERMS", ai_patterns)
    
    # Define financial patterns
    financial_patterns = [
        [{"LIKE_NUM": True}, {"LOWER": {"IN": ["billion", "million", "thousand"]}}, {"LOWER": "?", "OP": "?"}],
        [{"TEXT": {"REGEX": r"\$"}}, {"LIKE_NUM": True}],
        [{"LOWER": "revenue"}, {"LOWER": "of"}, {"TEXT": {"REGEX": r"\$"}}, {"LIKE_NUM": True}],
        [{"LOWER": {"IN": ["q1", "q2", "q3", "q4"]}}, {"LIKE_NUM": True}]
    ]
    
    matcher.add("FINANCIAL_TERMS", financial_patterns)
    
    # Define company/model names using PhraseMatcher
    company_terms = ["OpenAI", "Google DeepMind", "Meta", "Microsoft", "Tesla", "Apple"]
    model_terms = ["GPT-4", "GPT-3.5", "Claude", "Gemini", "LLaMA", "BERT", "T5"]
    
    company_patterns = [nlp(term) for term in company_terms]
    model_patterns = [nlp(term) for term in model_terms]
    
    phrase_matcher.add("COMPANIES", company_patterns)
    phrase_matcher.add("AI_MODELS", model_patterns)
    
    def extract_custom_patterns(text, nlp_model, matcher, phrase_matcher):
        """
        Extract custom patterns from text
        """
        doc = nlp_model(text)
        
        # Apply matchers
        matches = matcher(doc)
        phrase_matches = phrase_matcher(doc)
        
        # Organize results
        results = {
            'ai_terms': [],
            'financial_terms': [],
            'companies': [],
            'ai_models': []
        }
        
        # Process matcher results
        for match_id, start, end in matches:
            label = nlp_model.vocab.strings[match_id]
            span = doc[start:end]
            
            if label == "AI_TERMS":
                results['ai_terms'].append({
                    'text': span.text,
                    'start': start,
                    'end': end,
                    'tokens': [token.text for token in span]
                })
            elif label == "FINANCIAL_TERMS":
                results['financial_terms'].append({
                    'text': span.text,
                    'start': start,
                    'end': end,
                    'tokens': [token.text for token in span]
                })
        
        # Process phrase matcher results
        for match_id, start, end in phrase_matches:
            label = nlp_model.vocab.strings[match_id]
            span = doc[start:end]
            
            if label == "COMPANIES":
                results['companies'].append({
                    'text': span.text,
                    'start': start,
                    'end': end
                })
            elif label == "AI_MODELS":
                results['ai_models'].append({
                    'text': span.text,
                    'start': start,
                    'end': end
                })
        
        return results, doc
    
    # Test pattern matching on our sample texts
    for text_type, text in complex_texts.items():
        print(f"\n{'='*60}")
        print(f"PATTERN MATCHING: {text_type.upper()}")
        print(f"{'='*60}")
        
        results, doc = extract_custom_patterns(text, nlp, matcher, phrase_matcher)
        
        for pattern_type, matches in results.items():
            if matches:
                print(f"\n{pattern_type.upper().replace('_', ' ')}:")
                for match in matches:
                    print(f"  • {match['text']}")
                    if 'tokens' in match:
                        print(f"    Tokens: {match['tokens']}")
else:
    print("English model not available for pattern matching")

## 5. Entity Linking and Knowledge Base Integration

Connect entities to external knowledge sources (conceptual implementation).

In [ ]:
# Mock knowledge base for entity linking demonstration
knowledge_base = {
    'organizations': {
        'OpenAI': {
            'type': 'AI Research Company',
            'founded': '2015',
            'headquarters': 'San Francisco, CA',
            'key_products': ['GPT-4', 'ChatGPT', 'DALL-E'],
            'description': 'AI research and deployment company'
        },
        'Google DeepMind': {
            'type': 'AI Research Division',
            'founded': '2010 (DeepMind), 2023 (merged with Google AI)',
            'headquarters': 'London, UK',
            'key_products': ['Gemini', 'AlphaFold', 'AlphaGo'],
            'description': 'AI research division of Alphabet Inc.'
        },
        'Meta': {
            'type': 'Technology Company',
            'founded': '2004',
            'headquarters': 'Menlo Park, CA',
            'key_products': ['LLaMA', 'Facebook', 'Instagram', 'WhatsApp'],
            'description': 'Social media and technology conglomerate'
        }
    },
    'people': {
        'Sam Altman': {
            'role': 'CEO of OpenAI',
            'background': 'Former president of Y Combinator',
            'known_for': 'Leading OpenAI, entrepreneurship'
        },
        'Yann LeCun': {
            'role': 'Chief AI Scientist at Meta',
            'background': 'Professor at NYU, Turing Award winner',
            'known_for': 'Convolutional neural networks, deep learning'
        }
    },
    'locations': {
        'San Francisco': {
            'country': 'United States',
            'state': 'California',
            'known_for': 'Technology hub, Silicon Valley'
        },
        'London': {
            'country': 'United Kingdom',
            'known_for': 'Financial center, AI research hub'
        }
    },
    'models': {
        'GPT-4': {
            'developer': 'OpenAI',
            'type': 'Large Language Model',
            'release_date': '2023',
            'capabilities': ['Text generation', 'Reasoning', 'Code generation']
        },
        'Gemini': {
            'developer': 'Google DeepMind',
            'type': 'Multimodal AI Model',
            'release_date': '2023',
            'capabilities': ['Text', 'Image', 'Audio processing']
        }
    }
}

def entity_linking(entities, knowledge_base):
    """
    Link extracted entities to knowledge base entries
    """
    linked_entities = []
    
    for entity in entities:
        entity_text = entity['text']
        entity_label = entity['label']
        
        linked_info = None
        
        # Simple fuzzy matching (in practice, use more sophisticated methods)
        for category, kb_entries in knowledge_base.items():
            for kb_key, kb_info in kb_entries.items():
                if kb_key.lower() in entity_text.lower() or entity_text.lower() in kb_key.lower():
                    linked_info = {
                        'kb_category': category,
                        'kb_key': kb_key,
                        'kb_info': kb_info,
                        'confidence': 0.8  # Mock confidence score
                    }
                    break
            if linked_info:
                break
        
        linked_entity = {
            'original_entity': entity,
            'linked_info': linked_info
        }
        
        linked_entities.append(linked_entity)
    
    return linked_entities

if 'en_sm' in models:
    nlp = models['en_sm']
    
    # Perform entity linking on tech news text
    text = complex_texts['tech_news']
    doc = nlp(text)
    
    # Extract entities
    entities = []
    for ent in doc.ents:
        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'start': ent.start_char,
            'end': ent.end_char
        })
    
    # Perform entity linking
    linked_entities = entity_linking(entities, knowledge_base)
    
    print("ENTITY LINKING RESULTS:")
    print("="*60)
    
    for linked_entity in linked_entities:
        original = linked_entity['original_entity']
        linked = linked_entity['linked_info']
        
        print(f"\nEntity: '{original['text']}' ({original['label']})")
        
        if linked:
            print(f"✓ Linked to: {linked['kb_key']} ({linked['kb_category']})")
            print(f"  Confidence: {linked['confidence']}")
            print(f"  Info: {linked['kb_info']}")
        else:
            print(f"❌ No knowledge base match found")
else:
    print("English model not available for entity linking")

## 6. Custom Pipeline Components

Learn to build custom pipeline components for domain-specific processing.

In [ ]:
if 'en_sm' in models:
    from spacy.language import Language
    
    # Custom component 1: Technical Term Classifier
    @Language.component("technical_classifier")
    def technical_classifier(doc):
        """
        Custom component to classify technical terms
        """
        technical_terms = {
            'ai_ml': ['artificial intelligence', 'machine learning', 'deep learning', 'neural network', 
                     'transformer', 'attention', 'gradient', 'backpropagation'],
            'nlp': ['natural language processing', 'tokenization', 'embedding', 'semantic', 
                   'syntactic', 'parsing', 'lemmatization'],
            'data_science': ['data science', 'statistics', 'probability', 'regression', 
                           'classification', 'clustering', 'feature engineering'],
            'software': ['algorithm', 'optimization', 'scalability', 'api', 'framework', 
                        'library', 'deployment']
        }
        
        # Add custom attributes to tokens
        for token in doc:
            token._.is_technical = False
            token._.tech_category = None
        
        # Check for technical terms
        text_lower = doc.text.lower()
        for category, terms in technical_terms.items():
            for term in terms:
                if term in text_lower:
                    start_idx = text_lower.find(term)
                    # Find corresponding tokens (simplified)
                    for token in doc:
                        if token.idx >= start_idx and token.idx < start_idx + len(term):
                            token._.is_technical = True
                            token._.tech_category = category
        
        return doc
    
    # Custom component 2: Sentiment Scorer for Technical Text
    @Language.component("tech_sentiment")
    def tech_sentiment(doc):
        """
        Custom sentiment analysis for technical text
        """
        positive_indicators = ['improved', 'advanced', 'efficient', 'optimized', 'breakthrough', 
                              'innovative', 'superior', 'enhanced', 'significant']
        negative_indicators = ['limited', 'challenging', 'difficult', 'inefficient', 'problematic', 
                              'outdated', 'inferior', 'degraded']
        
        # Calculate sentiment score
        positive_score = sum(1 for token in doc if token.lemma_.lower() in positive_indicators)
        negative_score = sum(1 for token in doc if token.lemma_.lower() in negative_indicators)
        
        # Normalize by document length
        doc_length = len([token for token in doc if token.is_alpha])
        if doc_length > 0:
            sentiment_score = (positive_score - negative_score) / doc_length
        else:
            sentiment_score = 0.0
        
        # Add custom attribute to doc
        doc._.tech_sentiment = sentiment_score
        doc._.positive_terms = [token.text for token in doc if token.lemma_.lower() in positive_indicators]
        doc._.negative_terms = [token.text for token in doc if token.lemma_.lower() in negative_indicators]
        
        return doc
    
    # Register custom attributes
    if not Token.has_extension("is_technical"):
        Token.set_extension("is_technical", default=False)
    if not Token.has_extension("tech_category"):
        Token.set_extension("tech_category", default=None)
    if not Doc.has_extension("tech_sentiment"):
        Doc.set_extension("tech_sentiment", default=0.0)
    if not Doc.has_extension("positive_terms"):
        Doc.set_extension("positive_terms", default=[])
    if not Doc.has_extension("negative_terms"):
        Doc.set_extension("negative_terms", default=[])
    
    # Create custom pipeline
    nlp = models['en_sm']
    
    # Add custom components
    if "technical_classifier" not in nlp.pipe_names:
        nlp.add_pipe("technical_classifier", after="ner")
    if "tech_sentiment" not in nlp.pipe_names:
        nlp.add_pipe("tech_sentiment", last=True)
    
    print("CUSTOM PIPELINE CREATED:")
    print(f"Pipeline components: {nlp.pipe_names}")
    
    # Test custom pipeline
    test_texts = [
        "The new transformer architecture shows significant improvements in efficiency and accuracy.",
        "Traditional machine learning approaches are limited by their inability to handle complex patterns.",
        "Advanced neural networks demonstrate superior performance in natural language processing tasks."
    ]
    
    print("\nTESTING CUSTOM PIPELINE:")
    print("="*60)
    
    for i, text in enumerate(test_texts, 1):
        doc = nlp(text)
        
        print(f"\nText {i}: {text}")
        print(f"Technical sentiment: {doc._.tech_sentiment:.3f}")
        
        if doc._.positive_terms:
            print(f"Positive terms: {doc._.positive_terms}")
        if doc._.negative_terms:
            print(f"Negative terms: {doc._.negative_terms}")
        
        technical_tokens = [(token.text, token._.tech_category) 
                           for token in doc if token._.is_technical]
        if technical_tokens:
            print(f"Technical terms: {technical_tokens}")
else:
    print("English model not available for custom pipeline")

## 7. Integration with Modern Tokenizers

Demonstrate how to combine spaCy's linguistic analysis with modern tokenization.

In [ ]:
# Integration with modern tokenizers
try:
    import tiktoken
    from transformers import AutoTokenizer
    
    # Load modern tokenizers
    openai_tokenizer = tiktoken.encoding_for_model("gpt-4")
    hf_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    
    def hybrid_analysis(text, spacy_model, modern_tokenizers):
        """
        Combine spaCy linguistic analysis with modern tokenization
        """
        # spaCy analysis
        doc = spacy_model(text)
        
        # Extract linguistic features
        linguistic_features = {
            'entities': [(ent.text, ent.label_) for ent in doc.ents],
            'pos_tags': [(token.text, token.pos_) for token in doc if not token.is_punct],
            'dependencies': [(token.text, token.dep_, token.head.text) for token in doc if not token.is_punct],
            'sentences': [sent.text.strip() for sent in doc.sents],
            'lemmas': [(token.text, token.lemma_) for token in doc if not token.is_punct and not token.is_stop]
        }
        
        # Modern tokenization analysis
        tokenization_analysis = {}
        
        for name, tokenizer in modern_tokenizers.items():
            try:
                if hasattr(tokenizer, 'encode'):  # tiktoken
                    tokens = tokenizer.encode(text)
                    token_texts = [tokenizer.decode([t]) for t in tokens]
                else:  # HuggingFace
                    encoding = tokenizer(text, add_special_tokens=False)
                    tokens = encoding['input_ids']
                    token_texts = tokenizer.convert_ids_to_tokens(tokens)
                
                tokenization_analysis[name] = {
                    'token_count': len(tokens),
                    'tokens': token_texts[:15],  # First 15 tokens
                    'chars_per_token': len(text) / len(tokens) if tokens else 0
                }
            except:
                tokenization_analysis[name] = {'error': 'Failed to tokenize'}
        
        return {
            'text': text,
            'linguistic': linguistic_features,
            'tokenization': tokenization_analysis
        }
    
    # Test hybrid analysis
    if 'en_sm' in models:
        nlp = models['en_sm']
        
        sample_text = (
            "OpenAI's GPT-4 demonstrates remarkable improvements in natural language understanding. "
            "The model's advanced architecture enables sophisticated reasoning across diverse domains."
        )
        
        modern_tokenizers = {
            'gpt-4': openai_tokenizer,
            'bert-base': hf_tokenizer
        }
        
        analysis = hybrid_analysis(sample_text, nlp, modern_tokenizers)
        
        print("HYBRID ANALYSIS: spaCy + Modern Tokenizers")
        print("="*70)
        print(f"Text: {analysis['text']}")
        
        print(f"\nLINGUISTIC ANALYSIS (spaCy):")
        print(f"  Entities: {analysis['linguistic']['entities']}")
        print(f"  Sentences: {len(analysis['linguistic']['sentences'])}")
        print(f"  Key POS tags: {analysis['linguistic']['pos_tags'][:8]}")
        print(f"  Key lemmas: {analysis['linguistic']['lemmas'][:8]}")
        
        print(f"\nTOKENIZATION ANALYSIS:")
        for tokenizer_name, token_info in analysis['tokenization'].items():
            if 'error' not in token_info:
                print(f"  {tokenizer_name}:")
                print(f"    Token count: {token_info['token_count']}")
                print(f"    Chars/token: {token_info['chars_per_token']:.1f}")
                print(f"    Sample tokens: {token_info['tokens'][:8]}")
        
        # Create alignment analysis
        print(f"\nTOKEN ALIGNMENT ANALYSIS:")
        spacy_tokens = [token.text for token in nlp(sample_text) if not token.is_punct]
        print(f"  spaCy tokens ({len(spacy_tokens)}): {spacy_tokens[:10]}")
        
        for name, token_info in analysis['tokenization'].items():
            if 'error' not in token_info:
                print(f"  {name} tokens ({token_info['token_count']}): {token_info['tokens'][:10]}")
    else:
        print("English model not available for hybrid analysis")
        
except ImportError as e:
    print(f"Modern tokenizers not available: {e}")
    print("Install with: pip install tiktoken transformers")

## 8. Performance Optimization and Scaling

Techniques for optimizing spaCy performance in production environments.

In [ ]:
import time

def benchmark_spacy_performance(texts, nlp_model, iterations=10):
    """
    Benchmark spaCy performance with different configurations
    """
    results = {}
    
    # Full pipeline
    start_time = time.time()
    for _ in range(iterations):
        for text in texts:
            doc = nlp_model(text)
            # Access all annotations to ensure processing
            _ = list(doc.ents)
            _ = [(token.pos_, token.dep_) for token in doc]
    full_time = (time.time() - start_time) / iterations
    results['full_pipeline'] = full_time
    
    # Disable specific components for speed
    test_configs = [
        ('no_ner', ['ner']),
        ('no_parser', ['parser']),
        ('tokenizer_only', ['tagger', 'parser', 'ner'])
    ]
    
    for config_name, disabled_components in test_configs:
        with nlp_model.select_pipes(disable=disabled_components):
            start_time = time.time()
            for _ in range(iterations):
                for text in texts:
                    doc = nlp_model(text)
                    # Access available annotations
                    if 'ner' not in disabled_components:
                        _ = list(doc.ents)
                    if 'parser' not in disabled_components:
                        _ = [token.dep_ for token in doc]
            config_time = (time.time() - start_time) / iterations
            results[config_name] = config_time
    
    return results

def memory_efficient_processing(texts, nlp_model, batch_size=50):
    """
    Demonstrate memory-efficient batch processing
    """
    # Process in batches to manage memory
    all_results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Use nlp.pipe for efficient batch processing
        batch_results = []
        for doc in nlp_model.pipe(batch):
            # Extract only needed information to save memory
            result = {
                'text_length': len(doc.text),
                'token_count': len(doc),
                'entity_count': len(doc.ents),
                'sentence_count': len(list(doc.sents))
            }
            batch_results.append(result)
        
        all_results.extend(batch_results)
        
        # Optional: Clear cache between batches
        # gc.collect()
    
    return all_results

if 'en_sm' in models:
    nlp = models['en_sm']
    
    # Create test dataset
    base_texts = list(complex_texts.values())
    test_texts = base_texts * 5  # Multiply for more substantial testing
    
    print("PERFORMANCE OPTIMIZATION ANALYSIS")
    print("="*50)
    print(f"Testing with {len(test_texts)} texts, average length: {np.mean([len(t) for t in test_texts]):.0f} chars")
    
    # Benchmark different configurations
    perf_results = benchmark_spacy_performance(test_texts[:3], nlp, 5)  # Use subset for demo
    
    print(f"\nPERFORMANCE RESULTS (average time per iteration):")
    for config, time_taken in sorted(perf_results.items(), key=lambda x: x[1]):
        speedup = perf_results['full_pipeline'] / time_taken
        print(f"  {config:15} {time_taken:.3f}s (speedup: {speedup:.1f}x)")
    
    # Memory-efficient processing demo
    print(f"\nMEMORY-EFFICIENT PROCESSING:")
    efficient_results = memory_efficient_processing(test_texts, nlp, batch_size=3)
    
    print(f"  Processed {len(efficient_results)} documents")
    print(f"  Average tokens per doc: {np.mean([r['token_count'] for r in efficient_results]):.1f}")
    print(f"  Average entities per doc: {np.mean([r['entity_count'] for r in efficient_results]):.1f}")
    
    # Visualization of performance results
    plt.figure(figsize=(12, 6))
    
    # Performance comparison
    plt.subplot(1, 2, 1)
    configs = list(perf_results.keys())
    times = list(perf_results.values())
    
    bars = plt.bar(configs, times, color=['red' if 'full' in c else 'green' for c in configs])
    plt.title('spaCy Configuration Performance', fontweight='bold')
    plt.ylabel('Time (seconds)')
    plt.xlabel('Configuration')
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels
    for bar, time_val in zip(bars, times):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{time_val:.3f}s', ha='center', va='bottom', fontsize=9)
    
    # Document statistics
    plt.subplot(1, 2, 2)
    stats = ['token_count', 'entity_count', 'sentence_count']
    means = [np.mean([r[stat] for r in efficient_results]) for stat in stats]
    
    bars2 = plt.bar(stats, means, color=['blue', 'orange', 'purple'])
    plt.title('Average Document Statistics', fontweight='bold')
    plt.ylabel('Count')
    plt.xlabel('Statistic')
    
    # Add value labels
    for bar, val in zip(bars2, means):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
else:
    print("English model not available for performance testing")

## 9. Building a Complete NLP Pipeline

Integrate everything we've learned into a production-ready NLP pipeline.

In [ ]:
class AdvancedNLPPipeline:
    """
    Production-ready NLP pipeline combining classical and modern approaches
    """
    
    def __init__(self, language='en', model_size='sm'):
        self.language = language
        self.model_size = model_size
        
        # Load spaCy model
        model_name = f"{language}_core_web_{model_size}" if language == 'en' else f"{language}_core_news_{model_size}"
        try:
            self.nlp = spacy.load(model_name)
        except:
            print(f"Model {model_name} not found. Using blank model.")
            self.nlp = spacy.blank(language)
        
        # Initialize components
        self.matcher = Matcher(self.nlp.vocab)
        self.phrase_matcher = PhraseMatcher(self.nlp.vocab)
        
        # Setup custom patterns
        self._setup_patterns()
        
        # Add custom components if available
        self._setup_custom_components()
    
    def _setup_patterns(self):
        """Setup custom patterns for entity extraction"""
        # Technical terms patterns
        tech_patterns = [
            [{"LOWER": "artificial"}, {"LOWER": "intelligence"}],
            [{"LOWER": "machine"}, {"LOWER": "learning"}],
            [{"LOWER": "deep"}, {"LOWER": "learning"}],
            [{"LOWER": "natural"}, {"LOWER": "language"}, {"LOWER": "processing"}]
        ]
        self.matcher.add("TECH_TERMS", tech_patterns)
        
        # Financial patterns
        financial_patterns = [
            [{"TEXT": {"REGEX": r"\$"}}, {"LIKE_NUM": True}],
            [{"LIKE_NUM": True}, {"LOWER": {"IN": ["billion", "million", "thousand"]}}]
        ]
        self.matcher.add("FINANCIAL", financial_patterns)
    
    def _setup_custom_components(self):
        """Setup custom pipeline components"""
        # Only add if spaCy model is loaded and components don't exist
        if hasattr(self.nlp, 'pipe_names'):
            try:
                if "custom_analyzer" not in self.nlp.pipe_names:
                    @Language.component("custom_analyzer")
                    def custom_analyzer(doc):
                        # Add custom analysis here
                        return doc
                    
                    self.nlp.add_pipe("custom_analyzer", last=True)
            except:
                pass  # Skip if components can't be added
    
    def process_text(self, text, include_vectors=False):
        """
        Process text through the complete pipeline
        """
        doc = self.nlp(text)
        
        # Basic linguistic analysis
        result = {
            'text': text,
            'tokens': len(doc),
            'sentences': len(list(doc.sents)),
            'entities': [
                {
                    'text': ent.text,
                    'label': ent.label_,
                    'start': ent.start_char,
                    'end': ent.end_char,
                    'description': spacy.explain(ent.label_)
                } for ent in doc.ents
            ],
            'pos_tags': [
                {
                    'text': token.text,
                    'pos': token.pos_,
                    'lemma': token.lemma_
                } for token in doc if not token.is_punct and not token.is_space
            ][:20],  # Limit for display
            'custom_patterns': []
        }
        
        # Custom pattern matching
        matches = self.matcher(doc)
        for match_id, start, end in matches:
            label = self.nlp.vocab.strings[match_id]
            span = doc[start:end]
            result['custom_patterns'].append({
                'text': span.text,
                'label': label,
                'start': start,
                'end': end
            })
        
        # Add vectors if available and requested
        if include_vectors and self.nlp.vocab.vectors.size > 0:
            result['has_vectors'] = True
            result['vector_similarity'] = self._calculate_similarity(doc)
        else:
            result['has_vectors'] = False
        
        return result
    
    def _calculate_similarity(self, doc):
        """Calculate similarity metrics if vectors are available"""
        # This is a placeholder - in practice, you'd compare with reference documents
        if doc.vector.any():
            return {
                'vector_norm': float(np.linalg.norm(doc.vector)),
                'vector_dimensions': len(doc.vector)
            }
        return None
    
    def batch_process(self, texts, batch_size=50):
        """
        Process multiple texts efficiently
        """
        results = []
        
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            
            batch_results = []
            for doc in self.nlp.pipe(batch):
                # Simplified processing for batch mode
                result = {
                    'text_length': len(doc.text),
                    'tokens': len(doc),
                    'entities': len(doc.ents),
                    'sentences': len(list(doc.sents)),
                    'entity_types': list(set([ent.label_ for ent in doc.ents]))
                }
                batch_results.append(result)
            
            results.extend(batch_results)
        
        return results
    
    def get_pipeline_info(self):
        """Get information about the current pipeline"""
        return {
            'language': self.language,
            'model_size': self.model_size,
            'pipeline_components': self.nlp.pipe_names if hasattr(self.nlp, 'pipe_names') else [],
            'has_vectors': self.nlp.vocab.vectors.size > 0 if hasattr(self.nlp, 'vocab') else False,
            'vocab_size': len(self.nlp.vocab) if hasattr(self.nlp, 'vocab') else 0
        }

# Test the complete pipeline
if 'en_sm' in models:
    print("COMPLETE NLP PIPELINE DEMONSTRATION")
    print("="*60)
    
    # Initialize pipeline
    pipeline = AdvancedNLPPipeline('en', 'sm')
    
    # Display pipeline information
    info = pipeline.get_pipeline_info()
    print(f"Pipeline Info:")
    for key, value in info.items():
        print(f"  {key}: {value}")
    
    # Test single document processing
    test_text = complex_texts['tech_news'][:500]  # Use first 500 chars
    
    print(f"\nSINGLE DOCUMENT PROCESSING:")
    print(f"Input: {test_text[:100]}...")
    
    result = pipeline.process_text(test_text, include_vectors=True)
    
    print(f"\nResults:")
    print(f"  Tokens: {result['tokens']}")
    print(f"  Sentences: {result['sentences']}")
    print(f"  Entities: {len(result['entities'])}")
    print(f"  Custom patterns: {len(result['custom_patterns'])}")
    print(f"  Has vectors: {result['has_vectors']}")
    
    if result['entities']:
        print(f"\nTop entities:")
        for ent in result['entities'][:5]:
            print(f"  • {ent['text']} ({ent['label']}) - {ent['description']}")
    
    if result['custom_patterns']:
        print(f"\nCustom patterns found:")
        for pattern in result['custom_patterns'][:5]:
            print(f"  • {pattern['text']} ({pattern['label']})")
    
    # Test batch processing
    print(f"\nBATCH PROCESSING:")
    batch_texts = list(complex_texts.values())
    batch_results = pipeline.batch_process(batch_texts, batch_size=2)
    
    print(f"  Processed {len(batch_results)} documents")
    print(f"  Average tokens: {np.mean([r['tokens'] for r in batch_results]):.1f}")
    print(f"  Average entities: {np.mean([r['entities'] for r in batch_results]):.1f}")
    print(f"  Total unique entity types: {len(set().union(*[r['entity_types'] for r in batch_results]))}")
    
    # Summary statistics
    entity_type_counts = Counter()
    for result in batch_results:
        entity_type_counts.update(result['entity_types'])
    
    print(f"\nMost common entity types:")
    for entity_type, count in entity_type_counts.most_common(5):
        print(f"  {entity_type}: {count} documents")
else:
    print("Complete pipeline demonstration requires English spaCy model")

## 10. Key Takeaways and Best Practices

Summary of advanced spaCy capabilities and when to use them.

### Advanced spaCy Capabilities Summary

**Named Entity Recognition**:
- ✅ Pre-trained models for multiple languages
- ✅ High accuracy for standard entity types
- ✅ Easy visualization with displacy
- ⚠️ Limited to predefined entity types without custom training

**Dependency Parsing**:
- ✅ Detailed syntactic analysis
- ✅ Useful for understanding sentence structure
- ✅ Can identify subjects, objects, and relationships
- ⚠️ May be overkill for simple applications

**Custom Components**:
- ✅ Flexible extension of pipeline functionality
- ✅ Domain-specific processing capabilities
- ✅ Integration with existing pipeline
- ⚠️ Requires understanding of spaCy architecture

**Pattern Matching**:
- ✅ Powerful rule-based extraction
- ✅ Complementary to statistical approaches
- ✅ Good for domain-specific terms
- ⚠️ Maintenance overhead for complex rule sets

### When to Use Advanced spaCy Features

**Use Advanced NER When**:
- Need to extract structured information from text
- Working with well-defined entity types
- Building information extraction systems
- Require high precision on standard entities

**Use Dependency Parsing When**:
- Need to understand sentence structure
- Extracting relationships between entities
- Building question-answering systems
- Analyzing grammatical complexity

**Use Custom Components When**:
- Domain-specific requirements
- Need specialized processing logic
- Integrating with external systems
- Building production pipelines

### Integration with Modern Approaches

**Hybrid Strategies**:
1. **Preprocessing**: Use spaCy for cleaning, then modern tokenizers for model input
2. **Feature Engineering**: Extract linguistic features with spaCy for traditional ML
3. **Post-processing**: Use spaCy to structure outputs from language models
4. **Validation**: Use spaCy's linguistic analysis to validate model outputs

**Best Practices**:
1. **Choose the Right Model Size**: Balance accuracy vs. speed
2. **Disable Unused Components**: Improve performance by disabling unnecessary features
3. **Use Batch Processing**: More efficient for large document collections
4. **Cache Results**: Store processed results for repeated analysis
5. **Monitor Performance**: Track processing speed and memory usage
6. **Test with Domain Data**: Validate performance on your specific text types

### Performance Optimization Guidelines

**Speed Optimization**:
- Disable unused pipeline components
- Use smaller models when possible
- Process documents in batches
- Consider using spaCy's `select_pipes` context manager

**Memory Optimization**:
- Process documents in batches
- Clear document objects when done
- Use generators for large datasets
- Monitor memory usage in production

**Scalability Considerations**:
- Use multiprocessing for CPU-bound tasks
- Consider distributed processing for very large datasets
- Cache frequently accessed results
- Monitor and optimize bottlenecks

## 11. Exercise: Build Your Domain-Specific Pipeline

Create a custom NLP pipeline for a specific domain or use case.

In [ ]:
# Exercise: Design a domain-specific NLP pipeline

domains = [
    {
        'name': 'Legal Document Analysis',
        'requirements': [
            'Extract legal entities (cases, statutes, dates)',
            'Identify contract clauses and terms',
            'Analyze document structure',
            'Detect obligations and rights'
        ],
        'challenges': [
            'Complex legal terminology',
            'Long, nested sentences',
            'Domain-specific entity types',
            'Precision is critical'
        ]
    },
    {
        'name': 'Medical Text Processing',
        'requirements': [
            'Extract medical entities (drugs, symptoms, procedures)',
            'Identify patient information',
            'Analyze clinical notes structure',
            'Detect adverse events'
        ],
        'challenges': [
            'Medical abbreviations and terminology',
            'Privacy and compliance requirements',
            'Multiple languages/scripts',
            'High accuracy requirements'
        ]
    },
    {
        'name': 'Social Media Monitoring',
        'requirements': [
            'Real-time sentiment analysis',
            'Trend detection',
            'User mention and hashtag extraction',
            'Multi-language support'
        ],
        'challenges': [
            'Informal language and slang',
            'Emojis and special characters',
            'High volume processing',
            'Rapid language evolution'
        ]
    }
]

print("EXERCISE: Domain-Specific Pipeline Design")
print("="*50)

for i, domain in enumerate(domains, 1):
    print(f"\n{i}. {domain['name']}")
    print(f"\nRequirements:")
    for req in domain['requirements']:
        print(f"  • {req}")
    
    print(f"\nChallenges:")
    for challenge in domain['challenges']:
        print(f"  • {challenge}")
    
    print(f"\nYour Design Task:")
    print(f"  1. Which spaCy components would you use?")
    print(f"  2. What custom patterns would you create?")
    print(f"  3. How would you handle the specific challenges?")
    print(f"  4. What performance optimizations would you apply?")
    print(f"  5. How would you integrate with modern tokenizers?")
    
    print(f"\n  Your Pipeline Design:")
    print(f"  [Describe your approach here]")
    print(f"\n  " + "-"*50)

## Summary and Next Steps

You've now explored advanced spaCy features and learned how to:

- **Master NER**: Extract and analyze named entities with detailed statistics
- **Understand Syntax**: Use dependency parsing for structural analysis
- **Create Custom Components**: Build domain-specific processing logic
- **Pattern Matching**: Implement rule-based extraction systems
- **Optimize Performance**: Scale processing for production environments
- **Hybrid Integration**: Combine classical NLP with modern tokenization

### Recommended Next Steps

1. **Practice with Your Data**: Apply these techniques to your specific domain
2. **Explore spaCy Extensions**: Investigate additional libraries like spacy-transformers
3. **Custom Model Training**: Learn to train custom NER and classification models
4. **Production Deployment**: Study deployment patterns and monitoring
5. **Stay Updated**: Follow spaCy developments and new language models

### Resources for Continued Learning

- **Official Documentation**: [spaCy.io](https://spacy.io/)
- **Advanced Guide**: [spaCy 101: Everything you need to know](https://spacy.io/usage/spacy-101)
- **Custom Training**: [Training spaCy Models](https://spacy.io/usage/training)
- **Production Tips**: [spaCy Production Use](https://spacy.io/usage/production-use)
- **Community**: [spaCy Discussions](https://github.com/explosion/spaCy/discussions)

### Course Completion

Congratulations! You've completed the comprehensive 4-hour journey through:
- Classical text preprocessing fundamentals
- Modern tokenization strategies and comparisons  
- Advanced spaCy features and production techniques

You're now equipped to build sophisticated NLP systems that intelligently combine traditional linguistic analysis with cutting-edge language model technologies.